## Task 1: Compute Daily and Annualised Returns

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import os
import warnings

warnings.filterwarnings('ignore')

# Ensure output directories exist
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

# Load data
nav_history = pd.read_csv('../data/raw/02_nav_history.csv')
fund_master = pd.read_csv('../data/raw/01_fund_master.csv')
scheme_perf = pd.read_csv('../data/raw/07_scheme_performance.csv')
benchmarks = pd.read_csv('../data/raw/10_benchmark_indices.csv')

# Format dates
nav_history['date'] = pd.to_datetime(nav_history['date'])
benchmarks['date'] = pd.to_datetime(benchmarks['date'])

# Sort by date
nav_history = nav_history.sort_values(by=['amfi_code', 'date'])

# Task 1: Compute daily returns for all funds
nav_history['daily_return'] = nav_history.groupby('amfi_code')['nav'].pct_change()

# Compute annualised return: (1 + daily_return).prod()^(252/n) - 1
returns_summary = []
for amfi_code, group in nav_history.groupby('amfi_code'):
    group = group.dropna(subset=['daily_return'])
    if len(group) == 0:
        continue
    total_return = (1 + group['daily_return']).prod()
    n_days = len(group)
    ann_return = total_return ** (252 / n_days) - 1 if n_days > 0 else np.nan
    returns_summary.append({
        'amfi_code': amfi_code,
        'annualised_return': ann_return
    })

returns_df = pd.DataFrame(returns_summary)
returns_df.to_csv('../data/processed/returns_computed.csv', index=False)
print("Task 1 completed. Saved to returns_computed.csv")


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Task 1 completed. Saved to returns_computed.csv


## Task 2: Calculate CAGR for 1yr, 3yr, 5yr

In [2]:

# Task 2: Calculate CAGR for 1yr, 3yr, 5yr periods
max_date = nav_history['date'].max()

def get_cagr(group, years):
    start_date = max_date - pd.DateOffset(years=years)
    mask = group['date'] >= start_date
    period_data = group[mask]
    if len(period_data) < 200 * years: # Require sufficient data points
        return np.nan
    
    nav_start = period_data.iloc[0]['nav']
    nav_end = period_data.iloc[-1]['nav']
    return (nav_end / nav_start) ** (1 / years) - 1

cagr_results = []
for amfi_code, group in nav_history.groupby('amfi_code'):
    group = group.sort_values('date')
    cagr_1yr = get_cagr(group, 1)
    cagr_3yr = get_cagr(group, 3)
    cagr_5yr = get_cagr(group, 5)
    cagr_results.append({
        'amfi_code': amfi_code,
        'cagr_1yr': cagr_1yr,
        'cagr_3yr': cagr_3yr,
        'cagr_5yr': cagr_5yr
    })

cagr_df = pd.DataFrame(cagr_results)
cagr_df.to_csv('../data/processed/cagr_report.csv', index=False)
print("Task 2 completed. Saved to cagr_report.csv")


Task 2 completed. Saved to cagr_report.csv
